# 네이버 블로그 수집기 — 임신 관련 키워드 × 생활 위생 키워드

- **필수 키워드:** 임신 관련 키워드 목록
- **추가 포함 키워드:** 청소·세탁·식기·실내 환경·개인위생 키워드 목록
- **제외 키워드:** `협찬`, `제공받아`, `지원받아`, `소정의`, `원고료`
- **저장 형식:** JSONL


In [ ]:
# 필요한 패키지 설치 (최초 1회)
# !pip install selenium beautifulsoup4 requests tqdm pandas

In [2]:
from pathlib import Path
from urllib.parse import quote, urlparse, parse_qs, unquote
from datetime import datetime
import json
import re
import time

import requests as req
from bs4 import BeautifulSoup as bs
import pandas as pd
from tqdm.auto import tqdm

from selenium import webdriver as wb
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.options import Options
from selenium.common.exceptions import (
    NoSuchElementException,
    TimeoutException,
    WebDriverException,
)
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

c:\Users\6122\anaconda3\envs\py310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# =========================
# 1. 기본 설정
# =========================

PROJECT_DIR = Path(r"C:\Users\6122\Desktop\DXSchool_JKS\DX_Project\LG_DX_data-pipeline")
OUTPUT_DIR = PROJECT_DIR / "crawling_results"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

REQUIRED_KEYWORDS = list(dict.fromkeys([
    "임신", "임산부", "임신부", "임신 중", "임신초기", "임신중기", "임신 후기",
    "만삭", "예비맘", "입덧", "태동", "임테기", "산전", "출산 준비", "출산예정",
    "예비 엄마", "막달", "임신 주차", "예정일", "임신 몇주", "산모",
]))
INCLUDE_KEYWORDS = [
    "청소", "대청소", "청결", "곰팡이", "진드기", "닦기", "닦아", "닦다가", "닦고",
    "세탁", "빨래", "빨다", "빨아", "빨고", "빠는", "빨 때", "세탁세제", "섬유유연제",
    "이불", "건조", "설거지", "식기세척", "냉장고", "주방세제", "세척", "환기",
    "미세먼지", "공기청정기", "소독", "살균", "탈취", "냄새", "락스", "악취",
    "손소독제", "청결제", "샤워", "목욕", "속옷", "분비물", "세정제", "씻기", "씻다",
    "씻어", "씻었", "씻고", "씻는", "씻다가",
]
EXCLUDE_KEYWORDS = ["협찬", "제공받아", "지원받아", "소정의", "원고료"]

# 예전 코드의 시작일은 유지하고, 종료일은 현재 프로젝트 날짜로 설정
START_DATE = "20230101"
END_DATE = "20260902"

SCROLL_PAUSE_SEC = 1.0
PAGE_LOAD_SEC = 1.2
COMMENT_LOAD_SEC = 0.7
MAX_SCROLL_ROUNDS = 200   # 무한 스크롤 안전장치

RUN_TS = datetime.now().strftime("%Y%m%d_%H%M%S")
OUTPUT_PATH = OUTPUT_DIR / f"naver_blog_임신_위생키워드_{START_DATE}_{END_DATE}_{RUN_TS}.jsonl"

USER_AGENT = (
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
    "AppleWebKit/537.36 (KHTML, like Gecko) "
    "Chrome/147.0.0.0 Safari/537.36"
)

request_header = {"User-Agent": USER_AGENT}

print("결과 저장 경로:", OUTPUT_PATH)

결과 저장 경로: C:\Users\6122\Desktop\DXSchool_JKS\DX_Project\LG_DX_data-pipeline\crawling_results\naver_blog_임신_위생문제_20250101_20260901_20260901_163751.jsonl


In [4]:
# =========================
# 2. 검색어 조합 생성
# =========================
# 네이버 검색 문법 변화에 덜 민감하도록
# '필수 키워드 + 추가 포함 키워드' 모든 조합을 각각 검색하고 결과 URL을 합칩니다.
# 이후 본문 단계에서 제외 키워드를 다시 검증합니다.

SEARCH_QUERIES = [
    f'{required} {include}'
    for required in REQUIRED_KEYWORDS
    for include in INCLUDE_KEYWORDS
]

print(f"검색 조합 수: {len(SEARCH_QUERIES)}개")

임신 위생
임신 감염
임신 불편
임신 걱정
임신 문제
임신 살균
임신 소독
임신 냄새


In [5]:
# =========================
# 3. Selenium 드라이버
# =========================

def make_driver(headless=False, width=1400, height=1000):
    options = Options()
    options.add_argument(f"user-agent={USER_AGENT}")
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_argument("--disable-notifications")
    options.add_argument("--lang=ko-KR")
    options.add_argument(f"--window-size={width},{height}")
    if headless:
        options.add_argument("--headless=new")

    driver = wb.Chrome(options=options)
    driver.set_page_load_timeout(30)
    return driver


def build_blog_search_url(query, start_date=START_DATE, end_date=END_DATE):
    encoded = quote(query)
    return (
        "https://search.naver.com/search.naver"
        f"?ssc=tab.blog.all&query={encoded}"
        "&sm=tab_opt"
        f"&nso=so%3Ar%2Cp%3Afrom{start_date}to{end_date}"
    )

In [6]:
# =========================
# 4. 네이버 검색 결과에서 블로그 URL 수집
# =========================

def normalize_blog_url(url):
    """네이버 리다이렉트 URL이면 실제 URL을 최대한 복원합니다."""
    if not url:
        return None

    try:
        parsed = urlparse(url)
        qs = parse_qs(parsed.query)
        for key in ("url", "u", "target"):
            if key in qs and qs[key]:
                candidate = unquote(qs[key][0])
                if "blog.naver.com" in candidate:
                    return candidate
    except Exception:
        pass

    # 쿼리스트링/fragment/마지막 슬래시 차이로 같은 글이 중복되지 않게 정규화
    parsed = urlparse(url)
    return parsed._replace(query="", fragment="").geturl().rstrip("/")


def extract_blog_links(driver):
    links = set()

    # 특정 난수형 클래스 대신 href 중심으로 수집
    anchors = driver.find_elements(By.CSS_SELECTOR, 'a[href*="blog.naver.com"]')
    for a in anchors:
        href = normalize_blog_url(a.get_attribute("href"))
        if not href:
            continue
        if "blog.naver.com" not in href:
            continue
        # 홈/프로필성 URL 일부 제외
        if href.rstrip("/") in {"https://blog.naver.com", "http://blog.naver.com"}:
            continue
        links.add(href)

    return links


def collect_search_links(query):
    driver = make_driver(headless=False)
    collected = set()

    try:
        search_url = build_blog_search_url(query)
        print(f"\n[검색] {query}")
        driver.get(search_url)
        time.sleep(PAGE_LOAD_SEC)

        # '상세검색/더보기'는 DOM 변경이 잦으므로 있으면 클릭, 없으면 계속 진행
        for selector in [
            "a.more_link",
            "button.more_link",
            "a[class*='more']",
        ]:
            try:
                elem = driver.find_element(By.CSS_SELECTOR, selector)
                if elem.is_displayed():
                    driver.execute_script("arguments[0].click();", elem)
                    time.sleep(0.5)
                    break
            except Exception:
                pass

        last_height = driver.execute_script("return document.body.scrollHeight")
        stable_count = 0

        for _ in range(MAX_SCROLL_ROUNDS):
            collected.update(extract_blog_links(driver))

            driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
            time.sleep(SCROLL_PAUSE_SEC)

            new_height = driver.execute_script("return document.body.scrollHeight")
            if new_height == last_height:
                stable_count += 1
            else:
                stable_count = 0
                last_height = new_height

            # 3회 연속 높이 변화가 없으면 종료
            if stable_count >= 3:
                break

        collected.update(extract_blog_links(driver))
        print(f"링크 수집 완료: {len(collected)}개")

    finally:
        driver.quit()

    return collected

In [7]:
# =========================
# 5. 검색어별 URL 수집 및 중복 제거
# =========================

query_link_map = {}
all_links = set()

for query in SEARCH_QUERIES:
    links = collect_search_links(query)
    query_link_map[query] = sorted(links)
    all_links.update(links)

blog_href_list = sorted(all_links)
print("\n전체 중복 제거 링크 수:", len(blog_href_list))


[검색] 임신 위생
링크 수집 완료: 1600개

[검색] 임신 감염
링크 수집 완료: 1448개

[검색] 임신 불편
링크 수집 완료: 1506개

[검색] 임신 걱정
링크 수집 완료: 1450개

[검색] 임신 문제
링크 수집 완료: 1478개

[검색] 임신 살균
링크 수집 완료: 1717개

[검색] 임신 소독
링크 수집 완료: 1800개

[검색] 임신 냄새
링크 수집 완료: 1560개

전체 중복 제거 링크 수: 11115


In [8]:
# =========================
# 6. 블로그 본문/댓글 파싱 함수
# =========================

def safe_find_text(driver, selectors):
    for selector in selectors:
        try:
            elems = driver.find_elements(By.CSS_SELECTOR, selector)
            for elem in elems:
                txt = elem.text.strip()
                if txt:
                    return txt
        except Exception:
            pass
    return ""


def safe_find_all_text(driver, selectors):
    values = []
    for selector in selectors:
        try:
            elems = driver.find_elements(By.CSS_SELECTOR, selector)
            for elem in elems:
                txt = elem.text.strip()
                if txt:
                    values.append(txt)
            if values:
                break
        except Exception:
            pass
    return values


def switch_to_blog_frame(driver):
    driver.switch_to.default_content()
    try:
        driver.switch_to.frame("mainFrame")
        return True
    except Exception:
        return False


def click_comment_area(driver):
    # 네이버 블로그 버전에 따라 셀렉터가 달라질 수 있어 후보를 순차 시도
    selectors = [
        "span.btn_arr",
        "a.btn_comment",
        "button.btn_comment",
        "a[href*='CommentList']",
        "button[class*='comment']",
    ]

    for selector in selectors:
        try:
            elems = driver.find_elements(By.CSS_SELECTOR, selector)
            for elem in elems:
                if elem.is_displayed():
                    driver.execute_script("arguments[0].click();", elem)
                    time.sleep(COMMENT_LOAD_SEC)
                    return True
        except Exception:
            pass
    return False


def matched_include_keywords(text):
    return [kw for kw in INCLUDE_KEYWORDS if kw in text]


def matched_required_keywords(text):
    return [kw for kw in REQUIRED_KEYWORDS if kw in text]


def contains_excluded_keyword(text):
    return [kw for kw in EXCLUDE_KEYWORDS if kw in text]


def clean_text(value):
    """개행·이모티콘을 제거하고 공백 하나로 정규화합니다."""
    if value is None:
        return ""
    text = str(value)
    text = re.sub(r"[\r\n\t]+", " ", text)
    text = re.sub(r"[\U0001F1E6-\U0001F1FF\U0001F300-\U0001FAFF\u2600-\u27BF\uFE0F\u200D\u20E3]", "", text)
    return re.sub(r"\s+", " ", text).strip()


def collect_one_post(driver, url):
    driver.get(url)
    time.sleep(PAGE_LOAD_SEC)

    switch_to_blog_frame(driver)

    title = safe_find_text(driver, [
        "div.se-title-text span",
        "span.se-fs-",
        "h3.se_textarea",
        "div.pcol1 h3",
        "h3",
    ])

    content = safe_find_text(driver, [
        "div.se-main-container",
        "div#postViewArea",
        "div.post-view",
        "div.se_component_wrap",
    ])

    date = safe_find_text(driver, [
        "span.se_publishDate",
        "span.se_publishDate.pcol2",
        "p.date.fil5",
        "span.date",
    ])

    # 검색 키워드/광고성 제외어 판정을 위해 제목+본문을 함께 사용
    full_text = clean_text(f"{title} {content}")
    matched_required = matched_required_keywords(full_text)
    matched_keywords = matched_include_keywords(full_text)
    excluded_hits = contains_excluded_keyword(full_text)

    # 필수어와 추가어가 실제 글 내용에도 존재하는지 검증
    if not matched_required:
        return None, "required_keyword_missing"
    if not matched_keywords:
        return None, "include_keyword_missing"
    if excluded_hits:
        return None, f"excluded_keyword:{','.join(excluded_hits)}"

    comments = []
    try:
        click_comment_area(driver)
        body = driver.find_element(By.CSS_SELECTOR, "body")
        body.send_keys(Keys.END)
        time.sleep(COMMENT_LOAD_SEC)
        comments = safe_find_all_text(driver, [
            "span.u_cbox_contents",
            "div.u_cbox_text_wrap",
            ".u_cbox_text_wrap",
        ])
        comments = [clean_text(comment) for comment in comments]
        comments = [comment for comment in comments if comment]
    except Exception:
        comments = []

    record = {
        "required_keywords": REQUIRED_KEYWORDS,
        "matched_required_keywords": matched_required,
        "matched_keywords": matched_keywords,
        "title": clean_text(title),
        "date": date,
        "content": clean_text(content),
        "comments": comments,
        "comment_count": len(comments),
        "contents": clean_text(" ".join([title, content, *comments])),
        "url": url,
        "collector": "naver_blog_selenium",
        "collected_at": datetime.now().isoformat(timespec="seconds"),
    }

    return record, None

In [9]:
# =========================
# 7. 전체 글 수집 + JSONL 즉시 저장
# =========================
# 수집 중 중단되어도 이미 저장된 레코드는 남도록 한 줄씩 append 합니다.

records = []
failed = []

# 동일 파일명이 이미 있으면 새 실행 시 덮어쓰기
with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    driver = make_driver(headless=False, width=1080, height=900)
    driver.set_window_position(1200, 0)

    try:
        for url in tqdm(blog_href_list, desc="블로그 본문 수집"):
            try:
                record, reason = collect_one_post(driver, url)

                if record is None:
                    failed.append({"url": url, "reason": reason})
                    continue

                records.append(record)
                f.write(json.dumps(record, ensure_ascii=False) + "\n")
                f.flush()

            except Exception as e:
                failed.append({"url": url, "reason": type(e).__name__, "detail": str(e)[:300]})

    finally:
        driver.quit()

print(f"수집 성공: {len(records)}건")
print(f"제외/실패: {len(failed)}건")
print(f"저장 완료: {OUTPUT_PATH}")

블로그 본문 수집: 100%|██████████| 11115/11115 [15:19:36<00:00,  4.96s/it]  


수집 성공: 7159건
제외/실패: 3956건
저장 완료: C:\Users\6122\Desktop\DXSchool_JKS\DX_Project\LG_DX_data-pipeline\crawling_results\naver_blog_임신_위생문제_20250101_20260901_20260901_163751.jsonl


In [ ]:
# =========================
# 8. 결과 확인
# =========================

if records:
    preview_df = pd.DataFrame(records)
    display(preview_df[[
        "matched_required_keywords", "matched_keywords", "title", "date",
        "contents", "comment_count", "url", "collector"
    ]].head(20))
else:
    print("저장된 결과가 없습니다.")

if failed:
    failed_df = pd.DataFrame(failed)
    display(failed_df.head(20))

,required_keyword,matched_keywords,title,date,content,comments,comment_count,url,collector,collected_at
0,임신,"[걱정, 소독]",32주 차부터 분만까지 임신일기,2025. 12. 26. 8:10,ai생성이미지\n마지막 임신 일기다\n지금 우리 아기는 신생아를 막 졸업한\n생후 ...,[],0,https://blog.naver.com/001bao/224115159392,naver_blog_selenium,2026-09-01T18:20:23
1,임신,"[위생, 살균, 소독]",프로살림 휴대용 쪽쪽이 살균소독기 아기 외출 필수템 임신 출산선물,2026. 5. 8. 9:40,"휴대용 쪽쪽이 살균소독기\n아기 외출 필수템\n글, 사진 ⓒ 보떼\n😍\n안녕하세요...",[휴대용 쪽쪽이 살균소독기 아이랑 외출할때 필수템이네요!\n휴대성도 좋아보이고 여행...,8,https://blog.naver.com/01084558023/224278581089,naver_blog_selenium,2026-09-01T18:20:37
2,임신,"[위생, 불편]",임산부 목욕탕 가도 되나요?(온도 체크),2025. 6. 2. 0:58,임산부 목욕탕 가도 되나요?(온도 체크)\n안녕하세요. 워킹대디 만복파파입니다.\n...,[재밌는 글 감사합니다~ 또 놀러올게요ㅎㅎ],1,https://blog.naver.com/01190790097/223885451185,naver_blog_selenium,2026-09-01T18:20:47
3,임신,"[걱정, 문제]",상상임신 증상 임테기 두줄 가능한가요?,2025. 8. 10. 0:10,상상임신 원인 증상 임테기 두줄 가능한가요?\n안녕하세요. 워킹대디 만복파파입니다....,[],0,https://blog.naver.com/01190790097/223959317003,naver_blog_selenium,2026-09-01T18:20:52
4,임신,"[감염, 불편, 걱정]",임산부 배꼽 주변 통증 원인 이유 무엇인가요?,2025. 9. 24. 0:38,임산부 배꼽 주변 통증 원인 이유 무엇인가요?\n안녕하세요. 만복파파입니다.\n오늘...,[임신 중 배꼽 주변 통증이 이렇게 다양한 이유로 생길 수 있다니 신기해요. 특히 ...,4,https://blog.naver.com/01190790097/224019406770,naver_blog_selenium,2026-09-01T18:20:57
5,임신,"[불편, 문제]",임산부 발바닥 통증 치료 원인 완화 방법,2025. 9. 28. 0:10,임산부 발바닥 통증 치료 원인 완화 방법\n안녕하세요. 워킹대디 만복파파입니다.\n...,"[안녕하세요, 만복파파님.\n\n귀한 아기를 기다리는 임산부들의 고충을 정말 잘 설...",3,https://blog.naver.com/01190790097/224022424174,naver_blog_selenium,2026-09-01T18:21:02
6,임신,"[불편, 걱정, 문제, 냄새]",임산부 파스 냄새 맡아도 붙여도 되나요? 주의사항,2025. 9. 29. 0:10,임산부 파스 냄새 맡아도 붙여도 되나요? 주의사항\n안녕하세요. 워킹대디 만복파파입...,[임신 중 파스 사용에 대해 이렇게 상세히 설명해 주셔서 감사해요. 특히 멘톨이나 ...,4,https://blog.naver.com/01190790097/224022764908,naver_blog_selenium,2026-09-01T18:21:07
7,임신,"[위생, 감염, 걱정]",임신 초기 안정기 주수 언제부터?,2025. 12. 2. 0:30,임신 초기 안정기 주수 언제부터?\n안녕하세요. 워킹대디 만복파파입니다.\n임신을 ...,[임신 초기 안정기가 12주부터 시작된다는 점이 중요하군요! 이 시기에 태반이 자리...,2,https://blog.naver.com/01190790097/224081681201,naver_blog_selenium,2026-09-01T18:21:11
8,임신,"[불편, 문제]",임산부 다리 붓기 원인 빼는법 관리 방법 알아보기,2026. 5. 31. 0:20,임산부 다리 붓기 원인 빼는법 관리 방법 알아보기\n안녕하세요. 워킹대디 만복파파입...,"[🖐안녕하세요\n줄눈이빛나게입니다.\n정성스런글 잘보고 갑니다\n편안한밤 되세요, ...",2,https://blog.naver.com/01190790097/224297371854,naver_blog_selenium,2026-09-01T18:21:16
9,임신,"[불편, 걱정]",임신 15주-22주차 증상 / 식욕폭발 시기 / 꼬리뼈 통증 / 샛별이 태동,2026. 7. 4. 20:55,15주 증상기록\n잠 쏟아짐.\n배 속 더부룩. 찌르르싸르르\n변비증상 심해져 바나...,[],0,https://blog.naver.com/01250830/224336450995,naver_blog_selenium,2026-09-01T18:21:25


,url,reason,detail
0,https://blog.naver.com/001bao,required_keyword_missing,NaN
1,https://blog.naver.com/01084558023,required_keyword_missing,NaN
2,https://blog.naver.com/01190790097,required_keyword_missing,NaN
3,https://blog.naver.com/01250830/224142318198,excluded_keyword:지원받아,NaN
4,https://blog.naver.com/012yolo,required_keyword_missing,NaN
5,https://blog.naver.com/01oe032ds,required_keyword_missing,NaN
6,https://blog.naver.com/020102_,required_keyword_missing,NaN
7,https://blog.naver.com/0226_-,required_keyword_missing,NaN
8,https://blog.naver.com/0428smile,required_keyword_missing,NaN
9,https://blog.naver.com/0516kkj,required_keyword_missing,NaN


## JSONL 필드

| 필드 | 의미 |
|---|---|
| `required_keywords` | 검색에 사용한 필수 키워드 목록 |
| `matched_required_keywords` | 실제 제목/본문에서 확인된 필수 키워드 |
| `matched_keywords` | 실제 제목/본문에서 확인된 추가 포함 키워드 |
| `title` | 블로그 글 제목 |
| `date` | 게시일 |
| `content` | 본문 텍스트 |
| `comments` | 수집된 댓글 문자열 배열 |
| `comment_count` | 수집된 댓글 개수 |
| `contents` | 개행·이모티콘을 제거하고 제목+본문+댓글을 공백으로 합친 텍스트 |
| `url` | 원문 URL |
| `collector` | 수집 방식 식별자 |
| `collected_at` | 수집 시각 |

광고성 제외어(`협찬`, `제공받아`, `지원받아`, `소정의`, `원고료`)가 제목이나 본문에 포함된 글은 JSONL에 저장하지 않습니다.